In [ ]:
# === FaceScore filter check (paste into a Colab cell; no GPU needed) ===
# Question: how many extracted frames does the FaceScore >= 0.75 filter remove,
# in EVERY trial (not just the ones we inspected)?
# Reads the raw Py-Feat output files the pipeline saved (before any filtering).
import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/ThesisExperiment_cleann"
PARTICIPANTS = [1, 3, 4, 5, 6]
ROBOTS = ["Robot1", "Robot2", "Robot3", "Robot4", "Human"]
THRESHOLD = 0.75

from google.colab import drive
drive.mount("/content/drive")

rows = []
for pid in PARTICIPANTS:
    for robot in ROBOTS:
        path = os.path.join(BASE_DIR, f"participant_{pid}_{robot}_openface.csv")
        if not os.path.exists(path):
            print(f"[!] missing: {path}")
            continue
        df = pd.read_csv(path)
        if "FaceScore" not in df.columns:
            print(f"[!] no FaceScore column in {path}")
            continue
        s = df["FaceScore"]
        n = len(df)
        low = int((s < THRESHOLD).sum() + s.isna().sum())
        rows.append({"participant": pid, "robot": robot, "frames": n,
                     "removed": low, "pct_removed": round(100 * low / n, 2),
                     "min_FaceScore": round(s.min(), 3),
                     "mean_FaceScore": round(s.mean(), 3)})

res = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(res.to_string(index=False))
print("\n" + "=" * 60)
print(f"Trials checked: {len(res)} (expected 25)")
print(f"Total frames: {res['frames'].sum()}   removed: {res['removed'].sum()} "
      f"({100 * res['removed'].sum() / res['frames'].sum():.2f}%)")
print(f"Worst single trial: {res['pct_removed'].max():.2f}% removed")
res.to_csv(os.path.join(BASE_DIR, "facescore_filter_check.csv"), index=False)